# 02 — Telecom RAG demo and evaluation

This notebook builds the actual RAG system and evaluates whether retrieval helps.

The controlled experiment is:

```text
same LLM + same questions

LLM only
vs
LLM + retrieved telecom context
```

That isolates the effect of RAG better than comparing two different models.

We also evaluate **retrieval separately from generation**, because a RAG system can fail either by retrieving the wrong chunks or by generating a bad answer from good chunks.

## 0. Environment

Run after installing `requirements-dev.txt`. The notebook uses:
- LangChain for document/retrieval/LLM components,
- LangGraph for the conditional workflow,
- SentenceTransformers for free local embeddings,
- FAISS for vector search,
- Ollama + Qwen3 4B as the default free/local LLM.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

## 1. Download the RAG corpus

The corpus is intentionally small and high-quality rather than a random PDF dump. It contains:
1. ETSI/3GPP TS 38.215 for NR measurements,
2. ETSI/3GPP TS 38.214 for data procedures,
3. the AERPAW page describing the exact Ericsson dataset,
4. AERPAW post-processing documentation.

The files themselves are ignored by Git; `docs/SOURCES.md` and the downloader make the corpus reproducible.

In [ ]:
import subprocess

subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "download_docs.py")],
    check=False,
)

In [ ]:
from telecom_rag.config import DOCS_DIR

for path in sorted(DOCS_DIR.glob("*")):
    if path.is_file() and not path.name.startswith("."):
        print(f"{path.name:45s} {path.stat().st_size / 1024:8.1f} KB")

## 2. Load the documents

PDFs are extracted **page by page** with PyMuPDF so source metadata includes the PDF page. Web pages are stored as cleaned text. This metadata is later shown as citations in the UI.

In [ ]:
from telecom_rag.documents import load_documents

docs = load_documents()
print("Loaded document/page objects:", len(docs))
print("Example metadata:", docs[0].metadata)
print("\nExample text:\n", docs[0].page_content[:600])

## 3. Chunk the documents

Embedding a whole technical standard as one vector would be too coarse. We split documents into overlapping chunks.

Default:
- chunk size = 1200 characters
- overlap = 200 characters

Overlap reduces the chance that an important definition is cut exactly at a boundary.

In [ ]:
from telecom_rag.documents import chunk_documents

chunks = chunk_documents(docs)
print("Chunks:", len(chunks))
print("Example chunk metadata:", chunks[0].metadata)
print(chunks[0].page_content[:700])

## 4. Create local embeddings and the FAISS vector index

A SentenceTransformer maps each text chunk to a dense vector representing semantic meaning. FAISS stores those vectors and finds the chunks nearest to an embedded question.

The default embedding model is `sentence-transformers/all-MiniLM-L6-v2`, so this step does not require an API key.

In [ ]:
from telecom_rag.rag import get_embeddings
from langchain_community.vectorstores import FAISS

embeddings = get_embeddings()
store = FAISS.from_documents(chunks, embeddings)
print("Vectors in FAISS:", store.index.ntotal)

## 5. Save the index

Saving the index means Streamlit does not need to re-embed all documents on every startup.

In [ ]:
from telecom_rag.config import VECTOR_STORE_DIR

VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)
store.save_local(str(VECTOR_STORE_DIR))
print("Saved to:", VECTOR_STORE_DIR)

## 6. Inspect retrieval before using an LLM

This is important: **retrieval quality should be debugged independently of generation**.

If the right source is not in the top-k chunks, even a very capable LLM cannot ground its answer in that source.

In [ ]:
from telecom_rag.rag import retrieve

question = "What does RSRP measure in NR?"
retrieved = retrieve(store, question, k=4)

for i, doc in enumerate(retrieved, start=1):
    print(f"\n--- result {i} ---")
    print(doc.metadata)
    print(doc.page_content[:500].replace("\n", " "))

## 7. Choose the LLM

### Free/local default: Ollama

Install Ollama outside Python, then in a terminal:

```bash
ollama pull qwen3:4b
ollama serve
```

The notebook then connects through LangChain. This keeps the whole development demo local.

### Optional hosted provider

You can instead set `OPENAI_API_KEY` and call `get_llm(provider="openai")`. The RAG pipeline itself stays unchanged.

In [ ]:
from telecom_rag.rag import get_llm

PROVIDER = "ollama"       # change to "openai" if desired
MODEL = "qwen3:4b"        # for OpenAI you can use the repo's configured model

llm = get_llm(provider=PROVIDER, model=MODEL)
llm

## 8. Build the LangGraph workflow

The graph is intentionally small and useful:

```text
START
  ↓
route question
  ├── documentation question ─────────┐
  └── KPI-related + observation       │
              ↓                       │
         analyze KPI                  │
              └──────────→ retrieve ←─┘
                              ↓
                           generate
                              ↓
                             END
```

So “What is SINR?” does not waste time running KPI analysis, while a question about a selected observation can combine measured data with retrieved documentation.

In [ ]:
from telecom_rag.graph import build_graph

graph = build_graph(llm, store, reference_df=None, top_k=4)
graph

## 9. Documentation-only RAG demo

Here no KPI row is needed. LangGraph routes directly to retrieval and generation.

In [ ]:
result = graph.invoke({
    "question": "What do RSRP and SINR tell us about radio conditions?",
    "use_rag": True,
    "observation": None,
})

print("Route:", result["route"])
print("\nAnswer:\n", result["answer"])
print("\nSources:")
for source in result.get("sources", []):
    print(source)

## 10. KPI-aware demo

If Notebook 01 has been run, load the processed KPI table and select an unusual observation. The LLM receives:
1. the measured KPI values,
2. dataset-relative percentiles/anomaly context,
3. retrieved technical documentation.

The percentile statements are intentionally **relative to this dataset**, not universal RF-quality thresholds.

In [ ]:
from telecom_rag.config import PROCESSED_KPI_PATH
from telecom_rag.data import load_processed_kpis

if PROCESSED_KPI_PATH.exists():
    kpis = load_processed_kpis()
    if "anomaly_score" in kpis.columns:
        selected = kpis.sort_values("anomaly_score", ascending=False).iloc[0]
    else:
        selected = kpis.iloc[0]
    print(selected)
else:
    kpis = None
    selected = None
    print("Run Notebook 01 first to enable the KPI-aware demo.")

In [ ]:
if selected is not None:
    kpi_graph = build_graph(llm, store, reference_df=kpis, top_k=4)
    kpi_result = kpi_graph.invoke({
        "question": (
            "Why might this observation have this throughput, and which radio "
            "measurements are most relevant to investigate?"
        ),
        "use_rag": True,
        "observation": selected.to_dict(),
    })
    print("Route:", kpi_result["route"])
    print("\nData context supplied to the model:\n")
    print(kpi_result.get("kpi_context", ""))
    print("\nAnswer:\n", kpi_result["answer"])

## 11. Evaluation set

The repo includes a small, manually auditable benchmark in `eval/questions.json`.

Each example contains:
- a question,
- a short reference answer,
- required factual terms,
- the source document expected to contain the answer.

This is a **portfolio benchmark**, not a claim of production-grade telecom evaluation.

In [ ]:
from telecom_rag.evaluation import load_eval_questions

questions = load_eval_questions()
print("Questions:", len(questions))
questions[:2]

## 12. Evaluate retrieval with Hit@k

Before judging generated answers, ask a simpler question: **did retrieval include the expected source in the top k?**

This separates retrieval failures from generation failures.

In [ ]:
from telecom_rag.evaluation import evaluate_retrieval

retrieval_results = evaluate_retrieval(store, questions, k=4)
retrieval_results

In [ ]:
hit_col = "hit@4"
print(f"Retrieval {hit_col}: {retrieval_results[hit_col].mean():.1%}")

## 13. Same-LLM baseline: without RAG vs with RAG

Now we run every question twice using the **same model**:
- baseline: pretrained LLM only,
- RAG: same LLM plus top-k retrieved chunks.

We record:
- semantic similarity to the short reference answer,
- recall of required factual terms,
- citation presence,
- latency,
- whether the expected source was actually retrieved.

The default below uses five questions to keep local inference time reasonable. Set `limit=None` for the full benchmark.

In [ ]:
from telecom_rag.evaluation import compare_baseline_and_rag, summarize_comparison

results = compare_baseline_and_rag(
    llm=llm,
    store=store,
    embeddings=embeddings,
    questions=questions,
    k=4,
    limit=5,  # use None to run all questions
)
results[["id", "mode", "semantic_similarity", "required_term_recall",
         "citation_present", "latency_s", "retrieval_hit"]]

## 14. Aggregate the comparison

Do not assume RAG must win every metric. The purpose is to measure what changed. RAG may improve grounding/citations while adding latency; poor retrieval can also hurt the answer.

In [ ]:
summary = summarize_comparison(results)
summary

In [ ]:
plot_metrics = [c for c in ["semantic_similarity", "required_term_recall", "citation_present"]
                if c in summary.columns]

fig, ax = plt.subplots(figsize=(8, 5))
x = range(len(plot_metrics))
width = 0.35
modes = list(summary.index)

if len(modes) >= 2:
    ax.bar([i - width/2 for i in x], summary.loc[modes[0], plot_metrics], width, label=modes[0])
    ax.bar([i + width/2 for i in x], summary.loc[modes[1], plot_metrics], width, label=modes[1])
else:
    ax.bar(x, summary.loc[modes[0], plot_metrics], width)

ax.set_xticks(list(x), plot_metrics, rotation=20)
ax.set_ylim(0, 1)
ax.set_title("Same LLM: baseline vs RAG")
ax.legend()
plt.tight_layout()
plt.show()

## 15. Inspect failures, not just averages

A useful interview discussion is to identify *why* a question failed:
- expected source absent → retrieval problem,
- expected source present but answer poor → generation/prompt problem,
- answer correct without RAG → pretrained knowledge already sufficient,
- RAG answer cites evidence → grounding benefit even when factual score is similar.

In [ ]:
for qid in results["id"].unique():
    pair = results[results["id"] == qid]
    print("\n" + "=" * 80)
    print(qid, pair.iloc[0]["question"])
    for _, row in pair.iterrows():
        print(f"\n[{row['mode']}]")
        print(row["answer"][:900])

## 16. Useful next experiments

Once this baseline works, change one variable at a time:
- chunk size / overlap,
- top-k,
- embedding model,
- add a reranker,
- expand the troubleshooting corpus,
- compare retrieval queries with vs without KPI context,
- add more hand-authored benchmark cases.

Those ablations make the project look like **data science / ML experimentation**, not just an LLM wrapper.